In [ ]:
from tinysim.topdown_driving.widget import TopDownDrivingWidget

widget = TopDownDrivingWidget()
widget.render()

In [ ]:
from tinysim.topdown_driving.tk import TopDownDrivingTkFrontend
from tinysim.topdown_driving import TopDownDrivingEnv
import numpy as np
import asyncio

under = TopDownDrivingEnv(num_envs=2)
env = TopDownDrivingTkFrontend(sim_env=under)
env.show_rays = True

env.render()
car_index = 0

for _ in range(500):
    action = np.zeros((2, 2), dtype=np.float32)
    if "Up" in env.keys:
        action[car_index, 0] = 1.0
    if "Down" in env.keys:
        action[car_index, 0] = -1.0
    if "Left" in env.keys:
        action[car_index, 1] = -1.0
    if "Right" in env.keys:
        action[car_index, 1] = 1.0
    if "Shift_L" in env.keys:
        car_index = (car_index + 1) % env.sim_env.num_envs

    state = env.step(action, dt=0.02)
    await asyncio.sleep(0.02)

### Example Neural Network for controlling the car

Input: `state["rays"]`, 5 hit rays by distance, `[d0, d1, d2, d3, d4]`

Output layer with 2 neurons `[throttle, steer]`

In [ ]:
from tinysim.topdown_driving.tk import TopDownDrivingTkFrontend
from tinysim.topdown_driving import TopDownDrivingEnv
import numpy as np
import asyncio


class SmallNN:
    def __init__(self, in_dim=5, hidden=8, out_dim=2):
        self.w1 = np.random.randn(in_dim, hidden) * 0.5
        self.b1 = np.zeros(hidden)
        self.w2 = np.random.randn(hidden, out_dim) * 0.5
        self.b2 = np.zeros(out_dim)

    def forward(self, x):
        h = np.tanh(x @ self.w1 + self.b1)
        return np.tanh(h @ self.w2 + self.b2)


under = TopDownDrivingEnv(num_envs=25)
nets = [SmallNN() for _ in range(under.num_envs)]
env = TopDownDrivingTkFrontend(sim_env=under)
env.render()

action = np.zeros((under.num_envs, 2), dtype=np.float32)
state = env.step(action, dt=0.00)

for _ in range(600):
    for i, net in enumerate(nets):
        action[i] = net.forward(state["rays"][i])

    state = env.step(action, dt=0.02)
    await asyncio.sleep(0.02)